In [ ]:
# Step 1: Setup and Data Loading for CMI Sensor Data
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the datasets
train = pd.read_csv('/kaggle/input/competitions/cmi-detect-behavior-with-sensor-data/train.csv')
test = pd.read_csv('/kaggle/input/competitions/cmi-detect-behavior-with-sensor-data/test.csv')
train_demo = pd.read_csv('/kaggle/input/competitions/cmi-detect-behavior-with-sensor-data/train_demographics.csv')
test_demo = pd.read_csv('/kaggle/input/competitions/cmi-detect-behavior-with-sensor-data/test_demographics.csv')

# Inspect the shapes
print("=== Data Shapes ===")
print(f"Train time-series: {train.shape}")
print(f"Test time-series:  {test.shape}")
print(f"Train demographics: {train_demo.shape}")
print(f"Test demographics:  {test_demo.shape}")

# Inspect the training data columns
print("\n=== Train Data Columns (first 20) ===")
print(train.columns.tolist()[:20])

# Look at the target variable (gesture)
print("\n=== Target Variable Distribution (Gesture) ===")
print(train['gesture'].value_counts())

# Look at the first few rows of the training data
print("\n=== First 5 rows of Train Data (key columns) ===")
print(train[['row_id', 'sequence_id', 'sequence_counter', 'subject', 'gesture', 'behavior', 'acc_x', 'acc_y', 'acc_z']].head())

In [ ]:
# Step 2: Exploratory Data Analysis - Visualizing Sensor Signals
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

# --- 1. Check sequence lengths ---
print("=== Sequence Length Statistics ===")
seq_lengths = train.groupby('sequence_id').size()
print(f"Number of unique sequences: {len(seq_lengths)}")
print(f"Mean sequence length: {seq_lengths.mean():.1f} timesteps")
print(f"Median sequence length: {seq_lengths.median():.0f} timesteps")
print(f"Min sequence length: {seq_lengths.min()} timesteps")
print(f"Max sequence length: {seq_lengths.max()} timesteps")
print(f"\nDistribution of sequence lengths:")
print(seq_lengths.describe())

# --- 2. Pick one sequence for a BFRB gesture (e.g., "Cheek - pinch skin") ---
bfrb_gesture = "Cheek - pinch skin"
bfrb_seq_id = train[train['gesture'] == bfrb_gesture]['sequence_id'].iloc[0]
bfrb_seq = train[train['sequence_id'] == bfrb_seq_id].reset_index(drop=True)

# --- 3. Pick one sequence for a non-BFRB gesture (e.g., "Text on phone") ---
non_bfrb_gesture = "Text on phone"
non_bfrb_seq_id = train[train['gesture'] == non_bfrb_gesture]['sequence_id'].iloc[0]
non_bfrb_seq = train[train['sequence_id'] == non_bfrb_seq_id].reset_index(drop=True)

print(f"\n=== Selected Sequences ===")
print(f"BFRB: '{bfrb_gesture}' (Sequence: {bfrb_seq_id}, Length: {len(bfrb_seq)} timesteps)")
print(f"Non-BFRB: '{non_bfrb_gesture}' (Sequence: {non_bfrb_seq_id}, Length: {len(non_bfrb_seq)} timesteps)")

# --- 4. Plot IMU (Accelerometer) data for both sequences ---
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# BFRB sequence
axes[0, 0].plot(bfrb_seq['sequence_counter'], bfrb_seq['acc_x'], label='acc_x', color='red')
axes[0, 0].set_title(f'BFRB: {bfrb_gesture}\nAccelerometer X', fontsize=12)
axes[0, 0].set_ylabel('Acceleration (g)')

axes[0, 1].plot(bfrb_seq['sequence_counter'], bfrb_seq['acc_y'], label='acc_y', color='green')
axes[0, 1].set_title(f'Accelerometer Y', fontsize=12)

axes[0, 2].plot(bfrb_seq['sequence_counter'], bfrb_seq['acc_z'], label='acc_z', color='blue')
axes[0, 2].set_title(f'Accelerometer Z', fontsize=12)

# Non-BFRB sequence
axes[1, 0].plot(non_bfrb_seq['sequence_counter'], non_bfrb_seq['acc_x'], label='acc_x', color='red')
axes[1, 0].set_title(f'Non-BFRB: {non_bfrb_gesture}\nAccelerometer X', fontsize=12)
axes[1, 0].set_ylabel('Acceleration (g)')
axes[1, 0].set_xlabel('Timestep')

axes[1, 1].plot(non_bfrb_seq['sequence_counter'], non_bfrb_seq['acc_y'], label='acc_y', color='green')
axes[1, 1].set_title(f'Accelerometer Y', fontsize=12)
axes[1, 1].set_xlabel('Timestep')

axes[1, 2].plot(non_bfrb_seq['sequence_counter'], non_bfrb_seq['acc_z'], label='acc_z', color='blue')
axes[1, 2].set_title(f'Accelerometer Z', fontsize=12)
axes[1, 2].set_xlabel('Timestep')

plt.tight_layout()
plt.show()

# --- 5. Plot Thermopile data for both sequences ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Thermopiles for BFRB
for i in range(1, 6):
    axes[0].plot(bfrb_seq['sequence_counter'], bfrb_seq[f'thm_{i}'], label=f'thm_{i}')
axes[0].set_title(f'Thermopiles - BFRB: {bfrb_gesture}', fontsize=12)
axes[0].set_ylabel('Temperature')
axes[0].legend(loc='upper right', ncol=5)

# Thermopiles for Non-BFRB
for i in range(1, 6):
    axes[1].plot(non_bfrb_seq['sequence_counter'], non_bfrb_seq[f'thm_{i}'], label=f'thm_{i}')
axes[1].set_title(f'Thermopiles - Non-BFRB: {non_bfrb_gesture}', fontsize=12)
axes[1].set_ylabel('Temperature')
axes[1].set_xlabel('Timestep')
axes[1].legend(loc='upper right', ncol=5)

plt.tight_layout()
plt.show()

# --- 6. Plot a single Time-of-Flight (ToF) sensor channel ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot one ToF channel (tof_1_v0) for both sequences
axes[0].plot(bfrb_seq['sequence_counter'], bfrb_seq['tof_1_v0'], color='purple')
axes[0].set_title(f'ToF Sensor 1, Pixel 0 - BFRB: {bfrb_gesture}', fontsize=12)
axes[0].set_ylabel('Distance (mm)')

axes[1].plot(non_bfrb_seq['sequence_counter'], non_bfrb_seq['tof_1_v0'], color='orange')
axes[1].set_title(f'ToF Sensor 1, Pixel 0 - Non-BFRB: {non_bfrb_gesture}', fontsize=12)
axes[1].set_ylabel('Distance (mm)')
axes[1].set_xlabel('Timestep')

plt.tight_layout()
plt.show()

# --- 7. Check how many sequences per subject ---
print("\n=== Sequences per Subject ===")
seq_per_subject = train.groupby('subject')['sequence_id'].nunique().sort_values()
print(f"Average sequences per subject: {seq_per_subject.mean():.1f}")
print(f"Min sequences per subject: {seq_per_subject.min()}")
print(f"Max sequences per subject: {seq_per_subject.max()}")

# --- 8. Check demographics ---
print("\n=== Train Demographics ===")
print(train_demo.head())
print(f"\nAge distribution:\n{train_demo['age'].describe()}")
print(f"\nSex distribution:\n{train_demo['sex'].value_counts()}")
print(f"\nHandedness distribution:\n{train_demo['handedness'].value_counts()}")

In [ ]:
# Step 3: Sequence-Level Feature Extraction & Subject-Wise Splitting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("=== Step 3: Sequence-Level Feature Extraction ===\n")

# --- 1. Define which BFRB gestures are in our target ---
BFRB_GESTURES = [
    'Cheek - pinch skin', 'Neck - scratch', 'Eyebrow - pull hair',
    'Forehead - scratch', 'Forehead - pull hairline', 'Above ear - pull hair',
    'Neck - pinch skin', 'Eyelash - pull hair'
]
# All other gestures are "non-BFRB"

# --- 2. Create binary target ---
train['Is_BFRB'] = train['gesture'].isin(BFRB_GESTURES).astype(int)

# --- 3. Define feature groups ---
IMU_COLS = ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z']
THM_COLS = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
# ToF columns are 320 (5 sensors * 64 pixels). We'll aggregate them into 5 sensor-level features.
TOF_COLS = [col for col in train.columns if col.startswith('tof_')]

print(f"IMU columns: {len(IMU_COLS)}")
print(f"Thermopile columns: {len(THM_COLS)}")
print(f"ToF columns: {len(TOF_COLS)} (will aggregate to 5 sensor-level features)")

# --- 4. Aggregate each sequence into a fixed-length feature vector ---
print("\nAggregating sequences into feature vectors (this may take 1-2 minutes)...")

def extract_sequence_features(group):
    """Extract statistical features from a single sequence."""
    features = {}
    
    # IMU statistics
    for col in IMU_COLS:
        features[f'{col}_mean'] = group[col].mean()
        features[f'{col}_std'] = group[col].std()
        features[f'{col}_min'] = group[col].min()
        features[f'{col}_max'] = group[col].max()
    
    # Thermopile statistics
    for col in THM_COLS:
        features[f'{col}_mean'] = group[col].mean()
        features[f'{col}_std'] = group[col].std()
        features[f'{col}_min'] = group[col].min()
        features[f'{col}_max'] = group[col].max()
        # Range is important for temperature (how much did it change?)
        features[f'{col}_range'] = group[col].max() - group[col].min()
    
    # ToF: aggregate the 64 pixels per sensor into mean/std, then take stats over time
    for sensor_idx in range(1, 6):
        sensor_cols = [c for c in TOF_COLS if c.startswith(f'tof_{sensor_idx}_v')]
        if sensor_cols:
            # Spatial mean across pixels for each timestep
            spatial_mean = group[sensor_cols].mean(axis=1)
            features[f'tof_{sensor_idx}_mean'] = spatial_mean.mean()
            features[f'tof_{sensor_idx}_std'] = spatial_mean.std()
            features[f'tof_{sensor_idx}_min'] = spatial_mean.min()
            features[f'tof_{sensor_idx}_max'] = spatial_mean.max()
            features[f'tof_{sensor_idx}_range'] = spatial_mean.max() - spatial_mean.min()
    
    # Sequence length
    features['sequence_length'] = len(group)
    
    return pd.Series(features)

# Apply to training set
train_features = train.groupby('sequence_id').apply(extract_sequence_features).reset_index()
print(f"Extracted features shape: {train_features.shape}")

# --- 5. Merge with demographics and target labels ---
# Get one row per sequence for target and subject
seq_meta = train.groupby('sequence_id').agg({
    'subject': 'first',
    'gesture': 'first',
    'Is_BFRB': 'first'
}).reset_index()

# Merge features with metadata
train_features = train_features.merge(seq_meta, on='sequence_id', how='left')
train_features = train_features.merge(train_demo, on='subject', how='left')

print(f"Final training feature matrix: {train_features.shape}")
print(f"Columns: {train_features.columns.tolist()[:20]}...")

# --- 6. Prepare X and y ---
# Drop non-feature columns
drop_cols = ['sequence_id', 'subject', 'gesture', 'Is_BFRB']
feature_names = [c for c in train_features.columns if c not in drop_cols]

X = train_features[feature_names].fillna(0)
y_bfrb = train_features['Is_BFRB']  # Binary target
y_gesture = train_features['gesture']  # Multi-class target
groups = train_features['subject']  # For GroupKFold

print(f"\nFeature matrix shape: {X.shape}")
print(f"Binary target distribution:\n{y_bfrb.value_counts()}")

# --- 7. Subject-wise cross-validation ---
print("\n=== Subject-wise Cross-Validation (GroupKFold) ===\n")

gkf = GroupKFold(n_splits=5)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_bfrb, groups=groups)):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y_bfrb.iloc[train_idx], y_bfrb.iloc[val_idx]
    
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )
    model.fit(X_train_fold, y_train_fold)
    preds = model.predict(X_val_fold)
    
    # Binary F1 for BFRB detection
    f1 = f1_score(y_val_fold, preds)
    cv_scores.append(f1)
    print(f"Fold {fold+1}: Binary F1 = {f1:.4f} (Val subjects: {groups.iloc[val_idx].nunique()})")

print(f"\nMean Binary F1 across folds: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

# --- 8. Feature importance (top 15) ---
print("\n=== Top 15 Most Important Features ===")
model_full = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42, use_label_encoder=False)
model_full.fit(X, y_bfrb)
importance = pd.Series(model_full.feature_importances_, index=feature_names).sort_values(ascending=False)
print(importance.head(15))

# --- 9. Quick multi-class baseline (all 18 gestures) ---
print("\n=== Multi-class Baseline (18 gestures) ===\n")
le = LabelEncoder()
y_gesture_encoded = le.fit_transform(y_gesture)

gkf = GroupKFold(n_splits=5)
multi_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_gesture_encoded, groups=groups)):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y_gesture_encoded[train_idx], y_gesture_encoded[val_idx]
    
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='mlogloss', use_label_encoder=False
    )
    model.fit(X_train_fold, y_train_fold)
    preds = model.predict(X_val_fold)
    
    f1 = f1_score(y_val_fold, preds, average='macro')
    multi_f1_scores.append(f1)
    print(f"Fold {fold+1}: Macro F1 = {f1:.4f}")

print(f"\nMean Macro F1 across folds: {np.mean(multi_f1_scores):.4f} (+/- {np.std(multi_f1_scores):.4f})")

# --- 10. Save the feature matrix for later use ---
train_features.to_csv('train_sequence_features.csv', index=False)
print("\n✅ Saved: train_sequence_features.csv")

In [ ]:
# Definitive leakage test: shuffle the target labels
np.random.seed(42)
y_bfrb_shuffled = y_bfrb.sample(frac=1, random_state=42).reset_index(drop=True)

gkf = GroupKFold(n_splits=5)
shuffled_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_bfrb_shuffled, groups=groups)):
    model = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42, use_label_encoder=False)
    model.fit(X.iloc[train_idx], y_bfrb_shuffled.iloc[train_idx])
    preds = model.predict(X.iloc[val_idx])
    f1 = f1_score(y_bfrb_shuffled.iloc[val_idx], preds)
    shuffled_f1_scores.append(f1)

print(f"Shuffled Labels Binary F1: {np.mean(shuffled_f1_scores):.4f}")
print(f"Real Labels Binary F1: 0.9575")
print(f"Drop: {0.9575 - np.mean(shuffled_f1_scores):.4f}")

In [ ]:
# ============================================================================
# Step 4: NaN Handling + Stable Training
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# --- 1. DIAGNOSTIC: Check for NaN in raw data ---
print("\n=== NaN Diagnostic on Raw Sensor Data ===")
USED_FEATURES = ['acc_x','acc_y','acc_z','rot_w','rot_x','rot_y','rot_z',
                 'thm_1','thm_2','thm_3','thm_4','thm_5',
                 'tof_1_v27','tof_2_v27','tof_3_v27','tof_4_v27','tof_5_v27']
# Some columns might not exist; filter safely
available_features = [c for c in USED_FEATURES if c in train.columns]
print(f"Available features: {len(available_features)}")

nan_counts = train[available_features].isna().sum()
print(f"\nNaN counts per feature (only showing non-zero):")
print(nan_counts[nan_counts > 0] if nan_counts.sum() > 0 else "  ✅ No NaN in these columns")

# Also check for zeros (missing sensor readings)
zero_pct = (train[available_features] == 0).mean() * 100
print(f"\nZero percentage per feature:")
print(zero_pct.round(1))

# --- 2. Config ---
MAX_SEQ_LEN = 100
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 5e-4  # Reduced from 1e-3 for stability

train['Is_BFRB'] = train['gesture'].isin([
    'Cheek - pinch skin','Neck - scratch','Eyebrow - pull hair','Forehead - scratch',
    'Forehead - pull hairline','Above ear - pull hair','Neck - pinch skin','Eyelash - pull hair'
]).astype(int)

le = LabelEncoder()
train['gesture_encoded'] = le.fit_transform(train['gesture'])
BFRB_GESTURES = ['Cheek - pinch skin','Neck - scratch','Eyebrow - pull hair','Forehead - scratch',
                 'Forehead - pull hairline','Above ear - pull hair','Neck - pinch skin','Eyelash - pull hair']
bfrb_gestures_encoded = le.transform(BFRB_GESTURES)

# --- 3. FIXED Sequence Preparation ---
print("\nPreparing sequences (with NaN handling)...")
sequence_data, sequence_labels_gesture, sequence_subjects = [], [], []

for seq_id, group in train.groupby('sequence_id'):
    features = group[available_features].values.astype(np.float32)
    
    # ✅ FIX 1: Replace NaN with 0 BEFORE any computation
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Pad/truncate
    if len(features) >= MAX_SEQ_LEN:
        features = features[:MAX_SEQ_LEN]
    else:
        pad = np.zeros((MAX_SEQ_LEN - len(features), len(available_features)), dtype=np.float32)
        features = np.vstack([features, pad])
    
    # ✅ FIX 2: Clip std floor to prevent division by tiny values
    mean = features.mean(axis=0, keepdims=True)
    std = features.std(axis=0, keepdims=True)
    std = np.maximum(std, 1e-3)  # Floor std at 1e-3
    features = (features - mean) / std
    
    # ✅ FIX 3: Clip standardized values to [-10, 10] to prevent outliers
    features = np.clip(features, -10, 10)
    
    # Final NaN safety check
    if np.isnan(features).any():
        features = np.nan_to_num(features, nan=0.0)
    
    sequence_data.append(features.T)  # (C, T)
    sequence_labels_gesture.append(group['gesture_encoded'].iloc[0])
    sequence_subjects.append(group['subject'].iloc[0])

sequence_data = np.array(sequence_data, dtype=np.float32)
sequence_labels_gesture = np.array(sequence_labels_gesture)
sequence_subjects = np.array(sequence_subjects)

print(f"Sequence data shape: {sequence_data.shape}")
print(f"Any NaN in final data? {np.isnan(sequence_data).any()}")
print(f"Data range: [{sequence_data.min():.2f}, {sequence_data.max():.2f}]")

# --- 4. Dataset ---
class SensorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# --- 5. Model (simpler + more stable) ---
class SensorCNN(nn.Module):
    def __init__(self, n_channels, n_classes):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.3))
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1), nn.Dropout(0.3))
        self.fc = nn.Linear(128, n_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x).squeeze(-1)
        return self.fc(x)

# --- 6. Competition Metric ---
def competition_metric(y_true, y_pred, bfrb_enc):
    y_true_bfrb = np.isin(y_true, bfrb_enc).astype(int)
    y_pred_bfrb = np.isin(y_pred, bfrb_enc).astype(int)
    binary_f1 = f1_score(y_true_bfrb, y_pred_bfrb)
    bfrb_f1s = []
    for cls in bfrb_enc:
        t = (y_true == cls).astype(int)
        p = (y_pred == cls).astype(int)
        if t.sum() > 0:
            bfrb_f1s.append(f1_score(t, p, zero_division=0))
    macro_f1 = np.mean(bfrb_f1s) if bfrb_f1s else 0.0
    return 0.5 * binary_f1 + 0.5 * macro_f1, binary_f1, macro_f1

# --- 7. Training ---
print("\n=== Training FIXED CNN ===")
gkf = GroupKFold(n_splits=5)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(sequence_data, sequence_labels_gesture, groups=sequence_subjects)):
    print(f"\n--- Fold {fold+1}/5 ---")
    
    X_tr, y_tr = sequence_data[train_idx], sequence_labels_gesture[train_idx]
    X_va, y_va = sequence_data[val_idx], sequence_labels_gesture[val_idx]
    
    # ✅ FIX 4: Clip class weights to avoid explosion
    class_counts = np.bincount(y_tr, minlength=len(le.classes_)).astype(np.float32)
    class_weights = 1.0 / np.maximum(class_counts, 1.0)
    class_weights = class_weights / class_weights.sum() * len(le.classes_)
    class_weights = np.clip(class_weights, 0.1, 10.0)  # Cap weights
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
    
    train_loader = DataLoader(SensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(SensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
    
    model = SensorCNN(len(available_features), len(le.classes_)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits = model(X_b)
            loss = criterion(logits, y_b)
            
            # ✅ FIX 5: Check for NaN loss before backprop
            if torch.isnan(loss):
                print(f"  ⚠️ NaN loss detected at epoch {epoch+1}, skipping batch")
                optimizer.zero_grad()
                continue
            
            loss.backward()
            # ✅ FIX 6: Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        
        scheduler.step()
        
        # Validation
        model.eval()
        preds = []
        with torch.no_grad():
            for X_b, _ in val_loader:
                preds.extend(model(X_b.to(device)).argmax(1).cpu().numpy())
        preds = np.array(preds)
        score, bf1, mf1 = competition_metric(y_va, preds, bfrb_gestures_encoded)
        
        if (epoch+1) % 3 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d} | Loss: {train_loss/len(train_loader):.4f} | "
                  f"Score: {score:.4f} | Binary F1: {bf1:.4f} | Macro BFRB F1: {mf1:.4f}")
    
    fold_scores.append(score)
    print(f"  ✅ Fold {fold+1} Final: {score:.4f}")

print(f"\n=== Final: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f}) ===")

In [ ]:
# ============================================================================
# Step 5: CNN + Bi-LSTM for Temporal Modeling
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# --- 1. Use MORE ToF channels (add 2 more pixels per sensor) ---
USED_FEATURES = ['acc_x','acc_y','acc_z','rot_w','rot_x','rot_y','rot_z',
                 'thm_1','thm_2','thm_3','thm_4','thm_5']
# Add 3 ToF pixels per sensor (center, one above, one below)
for i in range(1, 6):
    for px in [19, 27, 35]:  # Three pixels per sensor
        col = f'tof_{i}_v{px}'
        if col in train.columns:
            USED_FEATURES.append(col)

available_features = [c for c in USED_FEATURES if c in train.columns]
print(f"Using {len(available_features)} channels")

MAX_SEQ_LEN = 100
BATCH_SIZE = 64
EPOCHS = 30  # Doubled from 15
LEARNING_RATE = 5e-4

# --- 2. Prepare sequences ---
train['Is_BFRB'] = train['gesture'].isin([
    'Cheek - pinch skin','Neck - scratch','Eyebrow - pull hair','Forehead - scratch',
    'Forehead - pull hairline','Above ear - pull hair','Neck - pinch skin','Eyelash - pull hair'
]).astype(int)

le = LabelEncoder()
train['gesture_encoded'] = le.fit_transform(train['gesture'])
BFRB_GESTURES = ['Cheek - pinch skin','Neck - scratch','Eyebrow - pull hair','Forehead - scratch',
                 'Forehead - pull hairline','Above ear - pull hair','Neck - pinch skin','Eyelash - pull hair']
bfrb_gestures_encoded = le.transform(BFRB_GESTURES)

print("Preparing sequences...")
sequence_data, sequence_labels_gesture, sequence_subjects = [], [], []

for seq_id, group in train.groupby('sequence_id'):
    features = group[available_features].values.astype(np.float32)
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    
    if len(features) >= MAX_SEQ_LEN:
        features = features[:MAX_SEQ_LEN]
    else:
        pad = np.zeros((MAX_SEQ_LEN - len(features), len(available_features)), dtype=np.float32)
        features = np.vstack([features, pad])
    
    mean = features.mean(axis=0, keepdims=True)
    std = np.maximum(features.std(axis=0, keepdims=True), 1e-3)
    features = np.clip((features - mean) / std, -10, 10)
    
    sequence_data.append(features.T)
    sequence_labels_gesture.append(group['gesture_encoded'].iloc[0])
    sequence_subjects.append(group['subject'].iloc[0])

sequence_data = np.array(sequence_data, dtype=np.float32)
sequence_labels_gesture = np.array(sequence_labels_gesture)
sequence_subjects = np.array(sequence_subjects)
print(f"Data shape: {sequence_data.shape}")

# --- 3. Dataset ---
class SensorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# --- 4. UPGRADED Model: CNN + Bi-LSTM ---
class CNNLSTM(nn.Module):
    def __init__(self, n_channels, n_classes, lstm_hidden=128):
        super().__init__()
        # CNN feature extractor (keeps temporal dimension)
        self.cnn = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.2),
            
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.3),
            
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
        )
        # After 2 max-pools: T=100 -> 50 -> 25
        # Bi-LSTM to capture temporal order
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=lstm_hidden,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )
        # Attention pooling over time
        self.attention = nn.Linear(lstm_hidden * 2, 1)
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, n_classes)
        )
    
    def forward(self, x):
        # x: (B, C, T)
        x = self.cnn(x)              # (B, 128, T')
        x = x.permute(0, 2, 1)       # (B, T', 128) for LSTM
        lstm_out, _ = self.lstm(x)   # (B, T', 256)
        
        # Attention pooling
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)  # (B, T', 1)
        pooled = (lstm_out * attn_weights).sum(dim=1)  # (B, 256)
        
        return self.fc(pooled)

# --- 5. Metric ---
def competition_metric(y_true, y_pred, bfrb_enc):
    y_true_bfrb = np.isin(y_true, bfrb_enc).astype(int)
    y_pred_bfrb = np.isin(y_pred, bfrb_enc).astype(int)
    binary_f1 = f1_score(y_true_bfrb, y_pred_bfrb)
    bfrb_f1s = []
    for cls in bfrb_enc:
        t = (y_true == cls).astype(int)
        p = (y_pred == cls).astype(int)
        if t.sum() > 0:
            bfrb_f1s.append(f1_score(t, p, zero_division=0))
    macro_f1 = np.mean(bfrb_f1s) if bfrb_f1s else 0.0
    return 0.5 * binary_f1 + 0.5 * macro_f1, binary_f1, macro_f1

# --- 6. Train ---
print("\n=== Training CNN + Bi-LSTM ===")
gkf = GroupKFold(n_splits=5)
fold_scores = []
fold_probs_store = []  # Save probabilities for later ensembling

for fold, (train_idx, val_idx) in enumerate(gkf.split(sequence_data, sequence_labels_gesture, groups=sequence_subjects)):
    print(f"\n--- Fold {fold+1}/5 ---")
    
    X_tr, y_tr = sequence_data[train_idx], sequence_labels_gesture[train_idx]
    X_va, y_va = sequence_data[val_idx], sequence_labels_gesture[val_idx]
    
    class_counts = np.bincount(y_tr, minlength=len(le.classes_)).astype(np.float32)
    class_weights = 1.0 / np.maximum(class_counts, 1.0)
    class_weights = class_weights / class_weights.sum() * len(le.classes_)
    class_weights = np.clip(class_weights, 0.1, 10.0)
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
    
    train_loader = DataLoader(SensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(SensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
    
    model = CNNLSTM(len(available_features), len(le.classes_)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)
    
    best_score = 0
    for epoch in range(EPOCHS):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        scheduler.step()
        
        # Evaluate every 5 epochs
        if (epoch+1) % 5 == 0 or epoch == 0:
            model.eval()
            preds = []
            with torch.no_grad():
                for X_b, _ in val_loader:
                    preds.extend(model(X_b.to(device)).argmax(1).cpu().numpy())
            preds = np.array(preds)
            score, bf1, mf1 = competition_metric(y_va, preds, bfrb_gestures_encoded)
            print(f"  Epoch {epoch+1:2d} | Score: {score:.4f} | Binary F1: {bf1:.4f} | Macro BFRB F1: {mf1:.4f}")
            best_score = max(best_score, score)
    
    fold_scores.append(best_score)
    print(f"  ✅ Fold {fold+1} Best: {best_score:.4f}")

print(f"\n=== Final: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f}) ===")
print(f"Previous CNN score: 0.6048")
print(f"Improvement: {np.mean(fold_scores) - 0.6048:+.4f}")

In [ ]:
# ============================================================================
# Step 6: Ensemble 5 CNN Folds + XGBoost for Final Validation Score
# ============================================================================
# This uses models already trained. No retraining needed.

import torch
import numpy as np
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

# --- 1. Re-train fold models and save them ---
print("=== Retraining folds and saving models for ensemble ===\n")
from torch.utils.data import DataLoader
from sklearn.model_selection import GroupKFold
import torch.optim as optim
import torch.nn as nn

# (Reuse SensorDataset, CNNLSTM, competition_metric from Step 5)
fold_models = []
gkf = GroupKFold(n_splits=5)
val_indices_list = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(sequence_data, sequence_labels_gesture, groups=sequence_subjects)):
    print(f"Training fold {fold+1}...")
    X_tr, y_tr = sequence_data[train_idx], sequence_labels_gesture[train_idx]
    
    class_counts = np.bincount(y_tr, minlength=len(le.classes_)).astype(np.float32)
    class_weights = 1.0 / np.maximum(class_counts, 1.0)
    class_weights = class_weights / class_weights.sum() * len(le.classes_)
    class_weights = np.clip(class_weights, 0.1, 10.0)
    cw_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
    
    train_loader = DataLoader(SensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    model = CNNLSTM(len(available_features), len(le.classes_)).to(device)
    opt = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
    crit = nn.CrossEntropyLoss(weight=cw_t)
    
    for epoch in range(20):  # 20 epochs for faster training
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = crit(model(X_b), y_b)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
        sched.step()
    
    fold_models.append(model)
    val_indices_list.append(val_idx)

# --- 2. Compute ensemble predictions on each fold's val set ---
print("\n=== Ensembling all 5 fold models ===\n")

# For each val fold, we ask ALL 5 models to predict, then average probabilities
all_true = []
all_pred_ensemble = []

for val_fold_idx, val_idx in enumerate(val_indices_list):
    X_val = sequence_data[val_idx]
    y_val = sequence_labels_gesture[val_idx]
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    
    # Collect probability predictions from all 5 fold models
    ensemble_probs = np.zeros((len(X_val), len(le.classes_)))
    for model in fold_models:
        model.eval()
        with torch.no_grad():
            probs = torch.softmax(model(X_val_t), dim=1).cpu().numpy()
        ensemble_probs += probs
    ensemble_probs /= len(fold_models)
    
    # Argmax for final prediction
    ensemble_preds = ensemble_probs.argmax(axis=1)
    
    all_true.extend(y_val)
    all_pred_ensemble.extend(ensemble_preds)

all_true = np.array(all_true)
all_pred_ensemble = np.array(all_pred_ensemble)

# --- 3. Compute competition metric for the ensemble ---
score, bf1, mf1 = competition_metric(all_true, all_pred_ensemble, bfrb_gestures_encoded)
print(f"🎯 ENSEMBLE VALIDATION SCORE: {score:.4f}")
print(f"   Binary F1:        {bf1:.4f}")
print(f"   Macro BFRB F1:    {mf1:.4f}")

print(f"\n📊 Comparison:")
print(f"   Single fold (avg): 0.6386")
print(f"   5-fold ensemble:   {score:.4f}")
print(f"   Improvement:       {score - 0.6386:+.4f}")

# --- 4. Save these trained models for later use (submission) ---
import pickle
with open('fold_models.pkl', 'wb') as f:
    pickle.dump({
        'models': fold_models,
        'le': le,
        'bfrb_gestures_encoded': bfrb_gestures_encoded,
        'available_features': available_features,
        'max_seq_len': MAX_SEQ_LEN
    }, f)
print("\n✅ Saved: fold_models.pkl (for submission API)")

In [ ]:
# ============================================================================
# Step 7: Compute Competition Metric for XGBoost, Then Blend with CNN
# ============================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# --- 1. XGBoost: get probabilities for all sequences using GroupKFold ---
print("=== Step 7a: Compute XGBoost Competition Metric (OOF) ===\n")

# X, y_gesture, groups are from Step 3
# But we need the encoder. Let's use the one we have.
le_gesture = LabelEncoder()
y_gesture_enc = le_gesture.fit_transform(train_features['gesture'])
bfrb_enc_local = le_gesture.transform(BFRB_GESTURES)

gkf = GroupKFold(n_splits=5)
xgb_oof_probs = np.zeros((len(X), len(le_gesture.classes_)))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y_gesture_enc, groups=groups)):
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85, random_state=42,
        eval_metric='mlogloss', use_label_encoder=False, 
        tree_method='hist'  # Faster
    )
    model.fit(X.iloc[tr_idx], y_gesture_enc[tr_idx])
    xgb_oof_probs[va_idx] = model.predict_proba(X.iloc[va_idx])

xgb_oof_preds = xgb_oof_probs.argmax(axis=1)

# Compute competition metric for XGBoost alone
def competition_metric(y_true, y_pred, bfrb_enc):
    y_true_bfrb = np.isin(y_true, bfrb_enc).astype(int)
    y_pred_bfrb = np.isin(y_pred, bfrb_enc).astype(int)
    binary_f1 = f1_score(y_true_bfrb, y_pred_bfrb)
    bfrb_f1s = []
    for cls in bfrb_enc:
        t = (y_true == cls).astype(int)
        p = (y_pred == cls).astype(int)
        if t.sum() > 0:
            bfrb_f1s.append(f1_score(t, p, zero_division=0))
    macro_f1 = np.mean(bfrb_f1s) if bfrb_f1s else 0.0
    return 0.5 * binary_f1 + 0.5 * macro_f1, binary_f1, macro_f1

xgb_score, xgb_bf1, xgb_mf1 = competition_metric(y_gesture_enc, xgb_oof_preds, bfrb_enc_local)
print(f"🎯 XGBoost Competition Score: {xgb_score:.4f}")
print(f"   Binary F1:        {xgb_bf1:.4f}")
print(f"   Macro BFRB F1:    {xgb_mf1:.4f}")

# --- 2. Retrain CNN fold models and get their OOF probabilities ---
print("\n=== Step 7b: Compute CNN Competition Metric (OOF) ===\n")

# Rebuild CNN data (same as before)
USED_FEATURES_CNN = ['acc_x','acc_y','acc_z','rot_w','rot_x','rot_y','rot_z',
                     'thm_1','thm_2','thm_3','thm_4','thm_5']
for i in range(1, 6):
    for px in [19, 27, 35]:
        col = f'tof_{i}_v{px}'
        if col in train.columns:
            USED_FEATURES_CNN.append(col)

available_features_CNN = [c for c in USED_FEATURES_CNN if c in train.columns]
MAX_SEQ_LEN = 100

train['Is_BFRB'] = train['gesture'].isin(BFRB_GESTURES).astype(int)
le_cnn = LabelEncoder()
train['gesture_encoded'] = le_cnn.fit_transform(train['gesture'])
bfrb_enc_cnn = le_cnn.transform(BFRB_GESTURES)

sequence_data, sequence_labels_gesture, sequence_subjects = [], [], []
for seq_id, group in train.groupby('sequence_id'):
    features = group[available_features_CNN].values.astype(np.float32)
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    if len(features) >= MAX_SEQ_LEN:
        features = features[:MAX_SEQ_LEN]
    else:
        pad = np.zeros((MAX_SEQ_LEN - len(features), len(available_features_CNN)), dtype=np.float32)
        features = np.vstack([features, pad])
    mean = features.mean(axis=0, keepdims=True)
    std = np.maximum(features.std(axis=0, keepdims=True), 1e-3)
    features = np.clip((features - mean) / std, -10, 10)
    sequence_data.append(features.T)
    sequence_labels_gesture.append(group['gesture_encoded'].iloc[0])
    sequence_subjects.append(group['subject'].iloc[0])

sequence_data = np.array(sequence_data, dtype=np.float32)
sequence_labels_gesture = np.array(sequence_labels_gesture)
sequence_subjects = np.array(sequence_subjects)

# (Reuse SensorDataset and CNNLSTM classes from Step 5)
class SensorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

gkf = GroupKFold(n_splits=5)
cnn_oof_probs = np.zeros((len(sequence_data), len(le_cnn.classes_)))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(sequence_data, sequence_labels_gesture, groups=sequence_subjects)):
    print(f"Training CNN fold {fold+1}/5...")
    X_tr, y_tr = sequence_data[tr_idx], sequence_labels_gesture[tr_idx]
    X_va = sequence_data[va_idx]
    
    cw = np.bincount(y_tr, minlength=len(le_cnn.classes_)).astype(np.float32)
    cw = 1.0 / np.maximum(cw, 1.0)
    cw = cw / cw.sum() * len(le_cnn.classes_)
    cw = np.clip(cw, 0.1, 10.0)
    cw_t = torch.tensor(cw, dtype=torch.float32).to(device)
    
    train_loader = DataLoader(SensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    model = CNNLSTM(len(available_features_CNN), len(le_cnn.classes_)).to(device)
    opt = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
    crit = nn.CrossEntropyLoss(weight=cw_t)
    
    for epoch in range(20):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = crit(model(X_b), y_b)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
        sched.step()
    
    # Predict on val
    model.eval()
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(X_va_t), dim=1).cpu().numpy()
    cnn_oof_probs[va_idx] = probs

cnn_oof_preds = cnn_oof_probs.argmax(axis=1)
cnn_score, cnn_bf1, cnn_mf1 = competition_metric(
    sequence_labels_gesture, cnn_oof_preds, bfrb_enc_cnn)
print(f"\n🎯 CNN Competition Score: {cnn_score:.4f}")
print(f"   Binary F1:        {cnn_bf1:.4f}")
print(f"   Macro BFRB F1:    {cnn_mf1:.4f}")

# --- 3. Blend XGBoost + CNN ---
print("\n=== Step 7c: Blending XGBoost + CNN ===\n")

# Both sets of probabilities are indexed by row order (sequence order).
# Verify they match
assert len(xgb_oof_probs) == len(cnn_oof_probs), "Length mismatch!"

# Try different blend weights
best_blend_score = 0
best_weight = 0
for w in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
    blended_probs = w * xgb_oof_probs + (1 - w) * cnn_oof_probs
    blended_preds = blended_probs.argmax(axis=1)
    score, bf1, mf1 = competition_metric(y_gesture_enc, blended_preds, bfrb_enc_local)
    print(f"  Weight XGB={w:.1f}, CNN={1-w:.1f} → Score: {score:.4f} | Binary F1: {bf1:.4f} | Macro BFRB F1: {mf1:.4f}")
    if score > best_blend_score:
        best_blend_score = score
        best_weight = w

print(f"\n🏆 BEST BLEND: {best_weight:.1f} × XGBoost + {1-best_weight:.1f} × CNN")
print(f"   Score: {best_blend_score:.4f}")

# --- 4. Summary Table ---
print("\n=== FINAL COMPARISON ===\n")
print(f"{'Model':<25} {'Score':<10} {'Binary F1':<12} {'Macro BFRB F1':<15}")
print("="*62)
print(f"{'XGBoost (statistical)':<25} {xgb_score:.4f}     {xgb_bf1:.4f}       {xgb_mf1:.4f}")
print(f"{'CNN + Bi-LSTM (raw)':<25} {cnn_score:.4f}     {cnn_bf1:.4f}       {cnn_mf1:.4f}")
print(f"{'Best Blend':<25} {best_blend_score:.4f}     {'-':<12} {'-':<15}")
print("="*62)
print(f"\n🥇 Overall Winner: ", end="")
if best_blend_score > max(xgb_score, cnn_score):
    print(f"BLEND (weight={best_weight:.1f})")
elif xgb_score > cnn_score:
    print("XGBoost")
else:
    print("CNN")

In [ ]:
# ============================================================================
# VIZ 1: Signal Patterns - BFRB vs Non-BFRB
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

# Pick one clear BFRB sequence and one clear non-BFRB sequence
bfrb_gesture = "Cheek - pinch skin"
non_bfrb_gesture = "Text on phone"

bfrb_seq = train[train['gesture'] == bfrb_gesture].groupby('sequence_id').first().reset_index()
bfrb_seq_id = bfrb_seq['sequence_id'].iloc[0]
bfrb_data = train[train['sequence_id'] == bfrb_seq_id].reset_index(drop=True)

non_seq = train[train['gesture'] == non_bfrb_gesture].groupby('sequence_id').first().reset_index()
non_seq_id = non_seq['sequence_id'].iloc[0]
non_data = train[train['sequence_id'] == non_seq_id].reset_index(drop=True)

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# Row 1: Accelerometer
for i, (col, color) in enumerate(zip(['acc_x', 'acc_y'], ['crimson', 'forestgreen'])):
    axes[0, i].plot(bfrb_data['sequence_counter'], bfrb_data[col], color=color, linewidth=1.8, label='BFRB')
    axes[0, i].plot(non_data['sequence_counter'], non_data[col], color='steelblue', linewidth=1.8, alpha=0.7, label='Non-BFRB')
    axes[0, i].set_title(f'{col.upper()} — Accelerometer', fontsize=12, fontweight='bold')
    axes[0, i].set_ylabel('Acceleration (g)')
    axes[0, i].legend(loc='best')
    axes[0, i].set_xlabel('Timestep')

# Row 2: Thermopile 2 (most important feature)
axes[1, 0].plot(bfrb_data['sequence_counter'], bfrb_data['thm_2'], color='darkorange', linewidth=2, label='BFRB')
axes[1, 0].plot(non_data['sequence_counter'], non_data['thm_2'], color='steelblue', linewidth=2, alpha=0.7, label='Non-BFRB')
axes[1, 0].set_title('THM_2 — Thermopile (Top Feature)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Temperature')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Timestep')

# ToF pixel distance
axes[1, 1].plot(bfrb_data['sequence_counter'], bfrb_data['tof_1_v27'], color='purple', linewidth=2, label='BFRB')
axes[1, 1].plot(non_data['sequence_counter'], non_data['tof_1_v27'], color='steelblue', linewidth=2, alpha=0.7, label='Non-BFRB')
axes[1, 1].set_title('TOF_1 Pixel 27 — Proximity', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Distance (mm)')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Timestep')

# Row 3: Bar chart comparing mean values across sensor groups
sensor_means_bfrb = [bfrb_data['acc_x'].std(), bfrb_data['acc_y'].std(), bfrb_data['acc_z'].std(),
                     bfrb_data['thm_2'].std(), bfrb_data['thm_3'].std(),
                     bfrb_data['tof_1_v27'].std(), bfrb_data['tof_2_v27'].std()]
sensor_means_non = [non_data['acc_x'].std(), non_data['acc_y'].std(), non_data['acc_z'].std(),
                    non_data['thm_2'].std(), non_data['thm_3'].std(),
                    non_data['tof_1_v27'].std(), non_data['tof_2_v27'].std()]
labels = ['acc_x', 'acc_y', 'acc_z', 'thm_2', 'thm_3', 'tof_1', 'tof_2']

x = np.arange(len(labels))
width = 0.35
axes[2, 0].bar(x - width/2, sensor_means_bfrb, width, label='BFRB', color='crimson', alpha=0.85)
axes[2, 0].bar(x + width/2, sensor_means_non, width, label='Non-BFRB', color='steelblue', alpha=0.85)
axes[2, 0].set_xticks(x)
axes[2, 0].set_xticklabels(labels)
axes[2, 0].set_title('Signal Variance Comparison', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Standard Deviation')
axes[2, 0].legend()

# Empty the last subplot or use it for a text summary
axes[2, 1].axis('off')
summary_text = (
    "Signal Characteristics:\n\n"
    "• BFRB gestures show sustained, rhythmic\n"
    "  accelerometer oscillations\n\n"
    "• Non-BFRB gestures show sharp, sudden\n"
    "  spikes (taps, movements)\n\n"
    "• Thermopile THM_2 shows large sustained\n"
    "  drop for BFRB (hand-to-face contact)\n\n"
    "• ToF sensors detect proximity changes\n"
    "  when hand nears face/neck"
)
axes[2, 1].text(0.05, 0.5, summary_text, fontsize=12, va='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

plt.suptitle('Multimodal Sensor Signatures: BFRB vs Non-BFRB Gestures',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('/kaggle/working/viz1_signal_patterns.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: viz1_signal_patterns.png")

In [ ]:
# ============================================================================
# VIZ 1b: Rescaled Signal Variance Comparison (Fixed Version)
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")

# Reuse data from viz1
# bfrb_data, non_data should still be in memory.
# If not, re-run the first few lines of viz1 to reload them.

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Panel 1: Accelerometer + Thermopile (small scale) ---
small_scale_labels = ['acc_x', 'acc_y', 'acc_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
bfrb_std_small = [bfrb_data[c].std() for c in small_scale_labels]
non_std_small = [non_data[c].std() for c in small_scale_labels]

x = np.arange(len(small_scale_labels))
width = 0.35

bars1 = axes[0].bar(x - width/2, bfrb_std_small, width, label='BFRB', color='crimson', alpha=0.85, edgecolor='black')
bars2 = axes[0].bar(x + width/2, non_std_small, width, label='Non-BFRB', color='steelblue', alpha=0.85, edgecolor='black')

axes[0].set_xticks(x)
axes[0].set_xticklabels(small_scale_labels, rotation=30, ha='right')
axes[0].set_ylabel('Standard Deviation', fontsize=12)
axes[0].set_title('Accelerometer & Thermopile Variance\n(Small-scale sensors)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.4)

# --- Panel 2: ToF Only (large scale) ---
tof_labels = ['tof_1_v27', 'tof_2_v27', 'tof_3_v27', 'tof_4_v27', 'tof_5_v27']
bfrb_std_tof = [bfrb_data[c].std() for c in tof_labels]
non_std_tof = [non_data[c].std() for c in tof_labels]

x = np.arange(len(tof_labels))
bars3 = axes[1].bar(x - width/2, bfrb_std_tof, width, label='BFRB', color='crimson', alpha=0.85, edgecolor='black')
bars4 = axes[1].bar(x + width/2, non_std_tof, width, label='Non-BFRB', color='steelblue', alpha=0.85, edgecolor='black')

axes[1].set_xticks(x)
axes[1].set_xticklabels([c.replace('_v27', '\n(pixel 27)') for c in tof_labels], fontsize=10)
axes[1].set_ylabel('Standard Deviation (mm)', fontsize=12)
axes[1].set_title('ToF Sensor Variance\n(Proximity sensors, mm scale)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.4)

plt.suptitle('Signal Variance Comparison — Separated by Sensor Scale',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/viz1b_rescaled_variance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: viz1b_rescaled_variance.png")

In [ ]:
# ============================================================================
# VIZ 2: Class Distribution & Dataset Overview
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set_style("whitegrid")
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

# --- Panel 1: Gesture distribution (18 classes) ---
ax1 = fig.add_subplot(gs[0, :2])
gesture_counts = train.groupby('sequence_id')['gesture'].first().value_counts()
colors = ['crimson' if g in BFRB_GESTURES else 'steelblue' for g in gesture_counts.index]
bars = ax1.barh(range(len(gesture_counts)), gesture_counts.values, color=colors, alpha=0.85)
ax1.set_yticks(range(len(gesture_counts)))
ax1.set_yticklabels(gesture_counts.index, fontsize=10)
ax1.invert_yaxis()
ax1.set_xlabel('Number of Sequences', fontsize=12)
ax1.set_title('Gesture Distribution (Red = BFRB, Blue = Non-BFRB)', fontsize=13, fontweight='bold')
for i, (bar, val) in enumerate(zip(bars, gesture_counts.values)):
    ax1.text(val + 30, i, str(val), va='center', fontsize=9)

# --- Panel 2: BFRB vs Non-BFRB pie chart ---
ax2 = fig.add_subplot(gs[0, 2])
bfrb_counts = train.groupby('sequence_id')['Is_BFRB'].first().value_counts()
ax2.pie(bfrb_counts.values, labels=['BFRB', 'Non-BFRB'], autopct='%1.1f%%',
        colors=['crimson', 'steelblue'], startangle=90, textprops={'fontsize': 12})
ax2.set_title('Binary Class Balance', fontsize=13, fontweight='bold')

# --- Panel 3: Age distribution ---
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(train_demo['age'], bins=20, color='mediumseagreen', alpha=0.85, edgecolor='black')
ax3.axvline(train_demo['age'].median(), color='red', linestyle='--', linewidth=2,
            label=f'Median: {train_demo["age"].median():.0f}')
ax3.set_xlabel('Age (years)', fontsize=12)
ax3.set_ylabel('Count of Subjects', fontsize=12)
ax3.set_title('Subject Age Distribution (n=81)', fontsize=13, fontweight='bold')
ax3.legend()

# --- Panel 4: Sex and handedness ---
ax4 = fig.add_subplot(gs[1, 1])
sex_counts = train_demo['sex'].map({0: 'Male', 1: 'Female'}).value_counts()
hand_counts = train_demo['handedness'].map({0: 'Left', 1: 'Right'}).value_counts()
x_pos = [0, 1]
ax4.bar([p - 0.2 for p in x_pos], [sex_counts.get('Male', 0), sex_counts.get('Female', 0)],
        width=0.35, label='Sex', color=['dodgerblue', 'hotpink'], alpha=0.85)
ax4.bar([p + 0.2 for p in x_pos], [hand_counts.get('Left', 0), hand_counts.get('Right', 0)],
        width=0.35, label='Handedness', color=['darkorange', 'purple'], alpha=0.85)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(['Group 1', 'Group 2'])
ax4.set_ylabel('Count', fontsize=12)
ax4.set_title('Sex & Handedness Distribution', fontsize=13, fontweight='bold')
ax4.legend(['Male / Left', 'Female / Right'], loc='upper right')

# --- Panel 5: Sequences per subject ---
ax5 = fig.add_subplot(gs[1, 2])
seq_per_subj = train.groupby('subject')['sequence_id'].nunique()
ax5.hist(seq_per_subj, bins=15, color='mediumpurple', alpha=0.85, edgecolor='black')
ax5.axvline(seq_per_subj.mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean: {seq_per_subj.mean():.0f}')
ax5.set_xlabel('Sequences per Subject', fontsize=12)
ax5.set_ylabel('Count', fontsize=12)
ax5.set_title('Sequences per Subject', fontsize=13, fontweight='bold')
ax5.legend()

fig.suptitle('CMI Dataset Overview — 8,151 Sequences from 81 Subjects, 18 Gestures',
             fontsize=16, fontweight='bold', y=0.98)
plt.savefig('/kaggle/working/viz2_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: viz2_class_distribution.png")

In [ ]:
# ============================================================================
# VIZ 3: Model Comparison — XGBoost vs CNN vs Blend
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_style("whitegrid")

# --- Actual scores from Step 7 ---
models = ['XGBoost\n(Statistical Features)', 'CNN + Bi-LSTM\n(Raw Time-Series)', 'Blend\n(0.6 XGB + 0.4 CNN)']
scores = [0.6999, 0.6142, 0.7066]
binary_f1 = [0.9475, 0.9252, 0.9517]
macro_f1 = [0.4522, 0.3032, 0.4615]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Panel 1: Overall Competition Score ---
colors = ['steelblue', 'darkorange', 'crimson']
bars1 = axes[0].bar(models, scores, color=colors, alpha=0.9, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Competition Score', fontsize=13)
axes[0].set_title('Overall Competition Score\n(0.5×Binary F1 + 0.5×Macro BFRB F1)', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 0.85)
axes[0].axhline(y=0.6999, color='steelblue', linestyle='--', alpha=0.5, linewidth=1)
for bar, val in zip(bars1, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.4f}',
                 ha='center', fontsize=12, fontweight='bold')
# Highlight winner
bars1[2].set_edgecolor('gold')
bars1[2].set_linewidth(3)

# --- Panel 2: Binary F1 ---
bars2 = axes[1].bar(models, binary_f1, color=colors, alpha=0.9, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Binary F1 Score', fontsize=13)
axes[1].set_title('Binary F1\n(BFRB vs Non-BFRB Detection)', fontsize=13, fontweight='bold')
axes[1].set_ylim(0.85, 1.0)
for bar, val in zip(bars2, binary_f1):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.4f}',
                 ha='center', fontsize=12, fontweight='bold')
# Highlight winner
bars2[2].set_edgecolor('gold')
bars2[2].set_linewidth(3)

# --- Panel 3: Macro BFRB F1 ---
bars3 = axes[2].bar(models, macro_f1, color=colors, alpha=0.9, edgecolor='black', linewidth=1.5)
axes[2].set_ylabel('Macro BFRB F1', fontsize=13)
axes[2].set_title('Macro BFRB F1\n(Per-Class Average over 8 BFRB Classes)', fontsize=13, fontweight='bold')
axes[2].set_ylim(0, 0.6)
for bar, val in zip(bars3, macro_f1):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.015, f'{val:.4f}',
                 ha='center', fontsize=12, fontweight='bold')
# Highlight winner
bars3[2].set_edgecolor('gold')
bars3[2].set_linewidth(3)

fig.suptitle('Model Performance Comparison — Blending Heterogeneous Models Wins',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/viz3_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: viz3_model_comparison.png")

In [ ]:
# ============================================================================
# VIZ 4: Blend Weight Sensitivity Curve
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_style("whitegrid")

# --- Your actual blend results from Step 7 ---
weights_xgb = [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
scores = [0.6142, 0.6612, 0.6831, 0.6952, 0.7023, 0.7066, 0.7060, 0.7033, 0.6999]
binary_f1 = [0.9252, 0.9346, 0.9406, 0.9462, 0.9503, 0.9517, 0.9521, 0.9515, 0.9475]
macro_f1 = [0.3032, 0.3877, 0.4255, 0.4442, 0.4542, 0.4615, 0.4599, 0.4550, 0.4522]

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(weights_xgb, scores, marker='o', markersize=10, linewidth=2.5,
        color='darkgreen', label='Overall Score', zorder=3)
ax.plot(weights_xgb, binary_f1, marker='s', markersize=8, linewidth=2,
        color='steelblue', label='Binary F1', alpha=0.7)
ax.plot(weights_xgb, macro_f1, marker='^', markersize=8, linewidth=2,
        color='crimson', label='Macro BFRB F1', alpha=0.7)

# Mark the winner
ax.scatter([0.6], [0.7066], s=300, facecolors='none', edgecolors='gold',
           linewidth=4, zorder=4, label='Optimal Weight (0.6)')
ax.axvline(x=0.6, color='gold', linestyle='--', alpha=0.5, linewidth=2)

# Annotations
ax.annotate('Peak: 0.7066', xy=(0.6, 0.7066), xytext=(0.7, 0.72),
            fontsize=12, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.set_xlabel('Weight on XGBoost (1 - weight on CNN)', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Blend Weight Sensitivity Analysis\nOptimal: 60% XGBoost + 40% CNN',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, alpha=0.4)
ax.set_xticks(weights_xgb)

plt.tight_layout()
plt.savefig('/kaggle/working/viz4_blend_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: viz4_blend_sensitivity.png")

In [ ]:
# ============================================================================
# VIZ 5: Confusion Matrix for the Best Blend Model
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

# --- Recreate the blend predictions ---
# We already computed xgb_oof_probs and cnn_oof_probs in Step 7.
blended_probs = 0.6 * xgb_oof_probs + 0.4 * cnn_oof_probs
blended_preds = blended_probs.argmax(axis=1)

# Use the gesture encoder from Step 7a
class_names = list(le_gesture.classes_)

# --- Full 18x18 confusion matrix (normalized by row) ---
cm = confusion_matrix(y_gesture_enc, blended_preds, normalize='true')

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Recall (Row-Normalized)'},
            annot_kws={'size': 8}, ax=ax, vmin=0, vmax=1)

ax.set_xlabel('Predicted Gesture', fontsize=13, fontweight='bold')
ax.set_ylabel('True Gesture', fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix — Blend Model (18 Gestures)\nNormalized by True Class (Recall)',
             fontsize=14, fontweight='bold', pad=15)

# Add a red box highlighting the BFRB block
n_bfrb = len(BFRB_GESTURES)
ax.add_patch(plt.Rectangle((0, 18-n_bfrb), 18, n_bfrb,
                            fill=False, edgecolor='red', lw=2.5))
ax.add_patch(plt.Rectangle((18-n_bfrb, 0), n_bfrb, 18,
                            fill=False, edgecolor='red', lw=2.5))

# Rotate x-labels for readability
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig('/kaggle/working/viz5_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: viz5_confusion_matrix.png")

# --- Also compute and print per-class recalls ---
print("\n=== Per-Class Recall (Blend Model) ===")
for i, name in enumerate(class_names):
    recall = cm[i, i]
    marker = "🔴 BFRB" if name in BFRB_GESTURES else "🔵 Non-BFRB"
    print(f"  {name:<45} Recall: {recall:.3f}  {marker}")

In [ ]:
# ============================================================================
# VIZ 6: XGBoost Feature Importance (Top 20)
# ============================================================================
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import seaborn as sns

sns.set_style("whitegrid")

# --- Retrain a single XGBoost on full data to get importances ---
le_g = LabelEncoder()
y_gesture_enc_full = le_g.fit_transform(train_features['gesture'])

model_full = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85, random_state=42,
    eval_metric='mlogloss', use_label_encoder=False, tree_method='hist'
)
model_full.fit(X, y_gesture_enc_full)

# --- Extract and plot top 20 ---
importance = pd.Series(model_full.feature_importances_, index=X.columns)
top20 = importance.sort_values(ascending=False).head(20)

# Color by feature group
def feature_color(name):
    if name.startswith('thm_'):
        return 'darkorange'
    elif name.startswith('tof_'):
        return 'purple'
    elif name.startswith('acc_'):
        return 'crimson'
    elif name.startswith('rot_'):
        return 'steelblue'
    else:
        return 'gray'

colors = [feature_color(name) for name in top20.index]

fig, ax = plt.subplots(figsize=(11, 9))
bars = ax.barh(range(len(top20)), top20.values, color=colors, alpha=0.9, edgecolor='black', linewidth=1)

ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Gain)', fontsize=13)
ax.set_title('Top 20 Feature Importances — XGBoost Model', fontsize=14, fontweight='bold')

# Annotate bars
for i, (bar, val) in enumerate(zip(bars, top20.values)):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='darkorange', label='Thermopile (Temperature)'),
    Patch(facecolor='purple', label='ToF (Proximity)'),
    Patch(facecolor='crimson', label='Accelerometer (Motion)'),
    Patch(facecolor='steelblue', label='Rotation (Orientation)'),
    Patch(facecolor='gray', label='Demographics / Other'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('/kaggle/working/viz6_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: viz6_feature_importance.png")

In [ ]:
# ============================================================================
# VIZ 7: Per-Class F1 Breakdown for BFRB Gestures
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score
import seaborn as sns

sns.set_style("whitegrid")

# --- Compute per-class F1 for all 3 models ---
bfrb_enc = le_gesture.transform(BFRB_GESTURES)

# XGBoost
xgb_preds = xgb_oof_probs.argmax(axis=1)
xgb_f1_per_class = []
for cls in bfrb_enc:
    t = (y_gesture_enc == cls).astype(int)
    p = (xgb_preds == cls).astype(int)
    xgb_f1_per_class.append(f1_score(t, p, zero_division=0))

# CNN
cnn_preds = cnn_oof_probs.argmax(axis=1)
cnn_f1_per_class = []
for cls in bfrb_enc:
    t = (sequence_labels_gesture == cls).astype(int)
    p = (cnn_preds == cls).astype(int)
    cnn_f1_per_class.append(f1_score(t, p, zero_division=0))

# Blend
blend_preds = (0.6 * xgb_oof_probs + 0.4 * cnn_oof_probs).argmax(axis=1)
blend_f1_per_class = []
for cls in bfrb_enc:
    t = (y_gesture_enc == cls).astype(int)
    p = (blend_preds == cls).astype(int)
    blend_f1_per_class.append(f1_score(t, p, zero_division=0))

# --- Plot ---
x = np.arange(len(BFRB_GESTURES))
width = 0.27

fig, ax = plt.subplots(figsize=(14, 7))

bars1 = ax.bar(x - width, xgb_f1_per_class, width, label='XGBoost', color='steelblue', alpha=0.9, edgecolor='black')
bars2 = ax.bar(x, cnn_f1_per_class, width, label='CNN + Bi-LSTM', color='darkorange', alpha=0.9, edgecolor='black')
bars3 = ax.bar(x + width, blend_f1_per_class, width, label='Blend (0.6/0.4)', color='crimson', alpha=0.9, edgecolor='black')

ax.set_ylabel('F1 Score', fontsize=13)
ax.set_title('Per-Class F1 Breakdown — 8 BFRB Gestures\n(Which gestures are easy vs hard to detect?)',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(BFRB_GESTURES, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 0.8)
ax.grid(True, axis='y', alpha=0.4)

# Annotate mean lines
ax.axhline(y=np.mean(xgb_f1_per_class), color='steelblue', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(y=np.mean(blend_f1_per_class), color='crimson', linestyle='--', alpha=0.5, linewidth=1)

# Add value labels on top of the blend bars
for bar, val in zip(bars3, blend_f1_per_class):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.2f}',
            ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/viz7_per_class_f1.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: viz7_per_class_f1.png")

# --- Print table for portfolio ---
print("\n=== Per-Class F1 Table ===")
print(f"{'Gesture':<30} {'XGBoost':<10} {'CNN':<10} {'Blend':<10}")
print("="*60)
for i, gesture in enumerate(BFRB_GESTURES):
    print(f"{gesture:<30} {xgb_f1_per_class[i]:<10.3f} {cnn_f1_per_class[i]:<10.3f} {blend_f1_per_class[i]:<10.3f}")
print("="*60)
print(f"{'Mean':<30} {np.mean(xgb_f1_per_class):<10.3f} {np.mean(cnn_f1_per_class):<10.3f} {np.mean(blend_f1_per_class):<10.3f}")

# 🧠 CMI - Detect Behavior with Sensor Data

[![Python](https://img.shields.io/badge/Python-3.8%2B-blue.svg)](https://www.python.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0-red.svg)](https://pytorch.org/)
[![XGBoost](https://img.shields.io/badge/XGBoost-2.0-orange.svg)](https://xgboost.readthedocs.io/)
[![Scikit-Learn](https://img.shields.io/badge/Scikit--Learn-1.3-green.svg)](https://scikit-learn.org/)
[![Kaggle](https://img.shields.io/badge/Kaggle-Child%20Mind%20Institute-20BEFF.svg)](https://www.kaggle.com/competitions/cmi-detect-behavior-with-sensor-data)

## 📖 Overview

This project tackles the Child Mind Institute (CMI) - Detect Behavior with Sensor Data Kaggle competition, a multimodal time-series classification task in the domain of mental health wearables.

**The goal:** Classify wrist-worn sensor data into 18 gesture classes**, distinguishing Body-Focused Repetitive Behaviors (BFRBs), such as hair pulling, skin pinching, and scratching, from everyday non-BFRB gestures. Early detection of BFRBs is clinically valuable for diagnosing and treating conditions like trichotillomania and excoriation disorder.

The data comes from the Helios device, which contains three sensor modalities:
- **IMU (7 channels):** 3-axis accelerometer + 4-channel orientation quaternion
- **Thermopiles (5 channels):** Infrared temperature sensors for detecting hand-to-skin contact
- **Time-of-Flight (5 × 8×8 grids):** Proximity sensors for distance measurement

### 🏆 Key Results

| Model | Competition Score | Binary F1 | Macro BFRB F1 |
|-------|------------------:|----------:|--------------:|
| XGBoost (statistical features) | 0.6999 | 0.9475 | 0.4522 |
| CNN + Bi-LSTM (raw time-series) | 0.6142 | 0.9252 | 0.3032 |
| **Blend (0.6 × XGB + 0.4 × CNN)** | **0.7066** 🏆 | 0.9517 | 0.4615 |

**Evaluation metric:** `0.5 × Binary F1 + 0.5 × Macro F1 over 8 BFRB classes`, computed via **subject-wise GroupKFold cross-validation** (81 subjects, no subject leakage).

---

## 💡 The Core Insight

> With only 81 subjects in the training set, a gradient-boosted tree on engineered statistical features outperformed a deep learning model on raw time-series (+0.086 score). However, the two models made complementary errors, and blending them yielded the best overall result.

This is a real-world ML lesson. Model complexity must be matched to data availability. Deep learning is powerful, but it needs diversity — and 81 subjects is not enough to learn robust temporal patterns from scratch. Feature engineering still wins when data is limited.

---

## 🔬 Methodology

### 1. Exploratory Data Analysis

I analyzed the physical signatures of BFRB vs. non-BFRB gestures across all three sensor modalities:

![Signal Patterns](images/viz1_signal_patterns.png)

**Observations:**
- BFRBs show sustained, rhythmic accelerometer oscillations and a sustained thermopile drop (hand contact cooling the sensor).
- Non-BFRBs show sharp, sudden spikes (phone taps) and erratic ToF readings.

![Rescaled Variance](images/viz1b_rescaled_variance.png)

### 2. Dataset Challenges

![Class Distribution](images/viz2_class_distribution.png)

- 8,151 sequences across 81 subjects (average 101 sequences per subject).
- 18 gesture classes with severe imbalance — BFRB classes have ~640 sequences each, while rarer non-BFRB gestures have only 161.
- 62.7% BFRB vs 37.3% non-BFRB binary split.
- Roughly 50% of test data will have missing thermopile/ToF sensors, forcing the model to generalize using IMU-only streams.

### 3. Feature Engineering (XGBoost Path)

For each of the 8,151 sequences, I extracted a 90-dimensional statistical feature vector by aggregating sensor signals over time:

- **IMU (7 channels × 4 stats)**: mean, std, min, max of acceleration and orientation
- **Thermopile (5 channels × 5 stats)**: mean, std, min, max, range (range captures the temperature drop during touch)
- **ToF (5 sensors × 5 stats)**: spatial mean across the 8×8 grid, then temporal mean/std/min/max/range
- **Demographics**: age, sex, handedness, height, shoulder-to-wrist, elbow-to-wrist

### 4. Deep Learning (CNN Path)

For the neural approach, each sequence was:
1. **Padded/truncated** to 100 timesteps
2. **Standardized** per sequence (zero mean, unit variance)
3. **Fed into a CNN + Bi-LSTM** with attention pooling:

Input (27 channels × 100 timesteps)
↓
Conv1D(64) → BN → ReLU → MaxPool → Dropout
↓
Conv1D(128) → BN → ReLU → MaxPool → Dropout
↓
Conv1D(128) → BN → ReLU → Dropout
↓
Bi-LSTM(128, 2 layers) → Attention Pooling
↓
FC(128) → FC(18 classes)


Trained with class weights, gradient clipping**, and cosine LR scheduling to handle imbalance and prevent NaN losses.

### 5. Blending Strategy

I swept the blend weight from 0.0 to 1.0 in steps of 0.1:

![Blend Sensitivity](images/viz4_blend_sensitivity.png)

The optimal weight is **0.6 × XGBoost + 0.4 × CNN**, which outperforms either model alone.

---

## 📊 Results

### Model Comparison

![Model Comparison](images/viz3_model_comparison.png)

### Confusion Matrix (Blend Model)

![Confusion Matrix](images/viz5_confusion_matrix.png)

**Key observations:**
- Non-BFRB gestures like "Text on phone" (0.94) and "Feel around in tray" (0.89) are almost perfectly classified.
- **Similar BFRB gestures confuse each other**: "Eyebrow - pull hair" vs "Forehead - pull hairline" share nearly identical arm trajectories and thermal signatures.

### Feature Importance (XGBoost)

![Feature Importance](images/viz6_feature_importance.png)

**The thermopile features dominate** — `thm_2_mean` and `thm_2_max` are the top two features. This validates the physical hypothesis: BFRB detection is fundamentally a touch detection problem, and thermopiles directly measure the thermal signature of hand-to-skin contact.

### Per-Class F1 Breakdown (BFRB Gestures)

![Per-Class F1](images/viz7_per_class_f1.png)

| BFRB Gesture | XGBoost | CNN + Bi-LSTM | Blend | Difficulty |
|--------------|--------:|--------------:|------:|:----------:|
| Above ear - pull hair | 0.685 | 0.483 | **0.697** | 🟢 Easy |
| Forehead - scratch | 0.577 | 0.450 | **0.590** | 🟢 Easy |
| Forehead - pull hairline | 0.487 | 0.233 | **0.494** | 🟡 Medium |
| Neck - scratch | 0.431 | 0.294 | **0.435** | 🟡 Medium |
| Cheek - pinch skin | 0.408 | 0.236 | **0.428** | 🟡 Medium |
| Neck - pinch skin | 0.367 | 0.325 | **0.389** | 🟠 Hard |
| Eyelash - pull hair | 0.361 | 0.249 | **0.365** | 🟠 Hard |
| Eyebrow - pull hair | 0.301 | 0.155 | 0.293 | 🔴 Very Hard |
| **Mean** | 0.452 | 0.303 | **0.461** | — |

---

## 🛠️ Technologies Used

- **Languages:** Python 3.8+
- **Deep Learning:** PyTorch (CNN + Bi-LSTM with attention)
- **Machine Learning:** XGBoost, Scikit-Learn
- **Data:** Pandas, NumPy
- **Visualization:** Matplotlib, Seaborn
- **Environment:** Kaggle Notebooks (GPU-accelerated)

---

## 📁 Project Structure
cmi-bfrb-detection/
├── notebook/
│ └── cmi_bfrb_detection.ipynb # Full Kaggle notebook
├── images/
│ ├── viz1_signal_patterns.png
│ ├── viz1b_rescaled_variance.png
│ ├── viz2_class_distribution.png
│ ├── viz3_model_comparison.png
│ ├── viz4_blend_sensitivity.png
│ ├── viz5_confusion_matrix.png
│ ├── viz6_feature_importance.png
│ └── viz7_per_class_f1.png
├── README.md
└── requirements.txt

## 🚀 How to Run

1. **Clone the repository:**
   ```bash
   git clone https://github.com/Adnyeus/cmi-bfrb-detection.git
   cd cmi-bfrb-detection

2. Install dependencies:
   ```bash
   pip install -r requirements.txt

4. Download the dataset from Kaggle and place it in the data/ folder.
5. Open the notebook:
   ```bash
   jupyter notebook notebook/cmi_bfrb_detection.ipynb

## 🔑 Key Takeaways

- Feature engineering still matters. With limited subjects (81), engineered statistical features beat raw time-series modeling by a wide margin.
- Blend heterogeneous models. XGBoost and CNN make different errors — their union beats either alone.
- Always compute the actual competition metric. The initial CNN's binary F1 looked great (0.95), but its macro BFRB F1 was poor (0.30). The competition metric reveals the truth.
- Physical intuition guides modeling. Feature importance confirmed that thermopile features (touch detection) drive performance — exactly as expected from the physical mechanism of BFRBs.

## 📝 Author
Ebad Naeem
[Github](https://github.com/Adnyeus) | [LinkedIn](https://www.linkedin.com/in/ebad-naeem-7984522b8)

## 🙏 Acknowledgments
- Kaggle and the Child Mind Institute for hosting the competition and providing the dataset.
- The open-source community for PyTorch, XGBoost, and Scikit-Learn.